In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
from langchain.agents import AgentState
class CustomState(AgentState):
    Favourite_color: str

In [ ]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def update_favourite_color(favourite_color:str, runtime: ToolRuntime) -> Command:
    """Update the favourite color in state as user revealed it"""
    return Command(update={
        "favourite_color": favourite_color,
        "messages":[ToolMessage("Successfully update the color", tool_call_id=tool)]}
)


In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model="gpt-5-nano",
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [ ]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="My favourite is Pink")]},
    {"configurable":{"thread_id":"1"}}
)

In [ ]:
from pprint import pprint
pprint(response)

In [ ]:
response = agent.invoke(
    {"messages":[HumanMessage(content="Hello how are you?")],
     "favourite_color":"green"},
    {"configurable":{"thread_id":"10"}}
)
print(response["messages"][-1].content)

In [ ]:
@tool
def read_custom_state(runtime:ToolRuntime) -> str:
    """Read the favourite color of user from state"""
    try:
        return runtime.state["favourite_color"]
    except:
        return "No favourite color in state"

agent = create_agent(
    model="gpt-5-nano",
    tools=[update_favourite_color, read_custom_state],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [ ]:
response = agent.invoke(
    {"messages":[HumanMessage(content="My favourite is Pink")]},
    {"configurable":{"thread_id":"10"}}
)
print(response)

In [ ]:
response=agent.invoke(
    {"messages":[HumanMessage(content="What is my favourite color?")]},
    {"configurable":{"thread_id": "10"}}
)
print(response)